# Домашнее задание №1: линейная регрессия (максимум 10 баллов)

Некоторые задания будут по вариантам (всего 4 варианта). Чтобы выяснить свой вариант, посчитайте количество букв в своей фамилии, возьмете остаток от деления на 4 и прибавьте 1.

**ФИО:** Монтлевич Дмитрий Константинович

**Вариант:** 2

In [1]:
import numpy as np

## Многомерная линейная регрессия из sklearn

Создадим набор данных для многомерной регрессии

In [2]:
from sklearn.datasets import make_regression

X, y = make_regression(n_samples = 10000)
print(X.shape, y.shape)

(10000, 100) (10000,)


У нас 10000 объектов и 100 признаков. Для начала решим задачу аналитически "из коробки".

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

reg1 = LinearRegression().fit(X, y)
print(mean_squared_error(y, reg1.predict(X)))


a=mean_squared_error(y, reg1.predict(X))

2.04423225125752e-25


Теперь попробуем обучить линейную регрессию методом градиентного спуска "из коробки"

In [4]:
from sklearn.linear_model import SGDRegressor

reg2 = SGDRegressor(alpha=0.00000001).fit(X, y)
print(mean_squared_error(y, reg2.predict(X)))

1.8408465499376074e-12


***Задание 1 (1 балл).*** Объясните, чем вызвана разница в значениях двух полученных значений метрики?

**Ответ:** Мы сравниваем два метода обучения линейной регрессии: обычную линейную регрессию **LinearRegression** и стохастический градиентный спуск **SGDRegressor**. Разницы в значениях метрик заключается в природе двух методов: аналитическое решение даёт оптимальный результат сразу, тогда как итеративные методы, такие как SGD, требуют времени для достижения оптимума, и могут не достигнуть его из-за ограничений, например, слишком малого шага градиента( в нашем случае alpha=0.00000001).

***Задание 2 (1 балл).*** Подберите гиперпараметры в методе градиентного спуска так, чтобы значение MSE было близко к значению MSE, полученному при обучении LinearRegression.

In [5]:
alpha=0.00000001

while True:
    reg2 = SGDRegressor(alpha=alpha).fit(X, y)
    b=mean_squared_error(y, reg2.predict(X))
    if b>a:
        alpha/=10
    else:
        print(f'Optimal alpha value is {alpha}')
        print(f'Best MSE is {mean_squared_error(y, reg2.predict(X))}')
        break  #нужно немного подождать

Optimal alpha value is 1.0000000000000002e-28
Best MSE is 1.6368707921175511e-25


## Ваша многомерная линейная регрессия

***Задание 3 (5 баллов)***. Напишите собственную многомерную линейную регрессию, оптимизирующую MSE методом *градиентного спуска*. Для этого используйте шаблонный класс.

Критерий остановки: либо норма разности весов на текущей и предыдущей итерациях меньше определенного значения (первый и третий варианты), либо модуль разности функционалов качества (MSE) на текущей и предыдущей итерациях меньше определенного значения (второй и четвертый варианты). Также предлагается завершать обучение в любом случае, если было произведено слишком много итераций.

***Задание 4 (2 балла)***. Добавьте l1 (первый и четвертый варианты) или l2 (второй и третий варианты) регуляризацию.

In [6]:
class LinearRegression(object):
    def __init__(self, alpha=0.1, l_ratio=0.0001, tol=0.0001, max_iter=1000):
        '''
        Для начала необходимо инициализировать параметры
        alpha - это learning rate или шаг обучения
        l_ratio - параметр регуляризации
        tol - значение для критерия останова
        max_iter - максимальное количество итераций обучения
        '''
        self.alpha = alpha 
        self.l_ratio = l_ratio
        self.tol = tol
        self.max_iter = max_iter
        

    def fit(self, X, y):
        X = np.insert(X, 0, 1, axis=1)
        w = np.random.randn(X.shape[1]) 
        for i in range(self.max_iter):
            y_pred = np.dot(X, w)
            mse = np.mean((y_pred - y) ** 2)
            gradient = (2 / y.shape[0]) * np.dot(X.T, (y_pred - y)) + self.l_ratio * w  
            w_new = w - self.alpha * gradient
            
            new_y_pred = np.dot(X, w_new)
            new_mse = np.mean((new_y_pred - y) ** 2)

            if np.abs(new_mse - mse) < self.tol:
                break
            w=w_new
        self.w = w

    def predict(self, X):
        X = np.insert(X, 0, 1, axis=1)
        return np.dot(X, self.w)

In [7]:
my_reg = LinearRegression()
my_reg.fit(X, y)
assert mean_squared_error(y, my_reg.predict(X)) < 1e-3
print('You are amazing! Great work!')

You are amazing! Great work!


***Задание 5 (1 балл)***. Обучите линейную регрессию из коробки с l1-регуляризацией (from sklearn.linear_model import Lasso, первый и четвертый варианты) или с l2-регуляризацией (from sklearn.linear_model import Ridge, второй и третий варианты) с значением параметра регуляризации 0.1. Обучите вашу линейную регрессию с тем же значением параметра регуляризации и сравните результаты. Сделайте выводы.

In [8]:
from sklearn.linear_model import Ridge
reg3 = Ridge(alpha=0.1).fit(X, y)
print(mean_squared_error(y, reg3.predict(X)))

1.9220894414924386e-06


In [9]:
reg4 = LinearRegression(alpha=0.1, l_ratio=0.0001, tol=0.0001, max_iter=1000)
reg4.fit(X, y)
print(mean_squared_error(y, reg4.predict(X)))

0.0003278084994804657


Модель Ridge, из коробки из sklearn.linear_model, на данных показала лучшее качество в плане среднеквадратичной ошибки (MSE), чем кастомная реализация LinearRegression, использующая l2-регуляризацию. Вероятно коробочка использует куда более оптимизированный алгоритм, чем написанный шаблонный клас, что приводит к значительной разнице